# Mini-Projeto ETL — IQVIA Data
## Fase 3: Carga com SCD Tipo 2 — Camada Gold (BigQuery)

**Objetivo:** Ler os dados tratados da camada Silver e populá-los no BigQuery aplicando **Slowly Changing Dimension Tipo 2 (SCD2)**.

### Lógica do SCD Tipo 2

| Situação | Ação |
|----------|------|
| Produto novo | Insere linha com `flag_ativo = TRUE` e `data_fim_validade = NULL` |
| Produto com valor alterado | Fecha linha anterior (`flag_ativo = FALSE`, `data_fim_validade = hoje`) e insere nova linha ativa |
| Produto sem alteração | Nenhuma ação |

### Colunas Obrigatórias na Tabela Gold

| Coluna | Descrição |
|--------|-----------|
| `sk_produto` | Chave substituta (Surrogate Key) — gerada via hash |
| `id_produto_original` | Chave natural: `ean` do produto |
| `valor_produto` | Volume SO Preço Popular (proxy de valor do produto) |
| `data_inicio_validade` | Data de início da validade do registro |
| `data_fim_validade` | Data de encerramento (`NULL` para o registro ativo) |
| `flag_ativo` | `TRUE` para registro atual, `FALSE` para histórico |

**Projeto GCP:** `proc-de-dados-iqvia-com-scd`  
**Dataset BigQuery:** `conj_dados_iqvia`  
**Tabela:** `dim_produto_scd2`

## 1. Instalação e Importação de Dependências

In [40]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


ERROR: Cannot install db-dtypes==1.5.0 and pandas==3.0.1 because these package versions have conflicting dependencies.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [41]:
import io
import hashlib
from datetime import datetime, timezone, date

import pandas as pd
from google.cloud import storage, bigquery
from google.cloud.bigquery import SchemaField

## 2. Configurações do Pipeline

In [42]:
GCS_PROJECT_ID    = "proc-de-dados-iqvia-com-scd"
GCS_BUCKET_NAME   = "etl-iqvia-data-lake-augusto"
GCS_SILVER_PREFIX = "silver/iqvia"

BQ_PROJECT_ID  = "proc-de-dados-iqvia-com-scd"
BQ_DATASET_ID  = "conj_dados_iqvia"
BQ_TABLE_ID    = "dim_produto_scd2"
BQ_TABLE_REF   = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}"

LOAD_TIMESTAMP = datetime.now(timezone.utc)
LOAD_DATE      = LOAD_TIMESTAMP.date()

# Partição Silver a ser lida. None = usa hoje.
SILVER_DATE_PARTITION = None
if SILVER_DATE_PARTITION is None:
    SILVER_DATE_PARTITION = LOAD_TIMESTAMP.strftime("%Y/%m/%d")

# Nome do arquivo Silver a consumir.
# Padrão: vendas_enriquecido.parquet
# Para simulação SCD2: vendas_enriquecido_v2.parquet
SILVER_FILE_NAME = "vendas_enriquecido_v2.parquet"

print(f"Projeto GCP    : {GCS_PROJECT_ID}")
print(f"Bucket GCS     : {GCS_BUCKET_NAME}")
print(f"Silver lida    : {GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/{SILVER_FILE_NAME}")
print(f"Tabela BigQuery: {BQ_TABLE_REF}")
print(f"Data carga     : {LOAD_DATE.strftime('%d/%m/%Y')}")
print(f"Timestamp      : {LOAD_TIMESTAMP.strftime('%d/%m/%Y %H:%M:%S UTC')}")

Projeto GCP    : proc-de-dados-iqvia-com-scd
Bucket GCS     : etl-iqvia-data-lake-augusto
Silver lida    : silver/iqvia/2026/03/20/vendas_enriquecido_v2.parquet
Tabela BigQuery: proc-de-dados-iqvia-com-scd.conj_dados_iqvia.dim_produto_scd2
Data carga     : 20/03/2026
Timestamp      : 20/03/2026 13:00:02 UTC


## 3. Conexão com GCS e BigQuery

In [43]:
# Autenticação via Application Default Credentials (ADC).
# Execute previamente no terminal (requer Google Cloud SDK instalado):
#   gcloud auth application-default login
#   gcloud auth application-default set-quota-project proc-de-dados-iqvia-com-scd

gcs_client = storage.Client(project=GCS_PROJECT_ID)
bq_client  = bigquery.Client(project=BQ_PROJECT_ID)

print(f"GCS conectado  | Projeto: {gcs_client.project}")
print(f"BQ conectado   | Projeto: {bq_client.project}")

GCS conectado  | Projeto: proc-de-dados-iqvia-com-scd
BQ conectado   | Projeto: proc-de-dados-iqvia-com-scd


## 4. Schema e Criação da Tabela Gold (se não existir)

In [44]:
GOLD_SCHEMA = [
    SchemaField("sk_produto",          "STRING",  mode="REQUIRED", description="Surrogate Key gerada via hash SHA256"),
    SchemaField("id_produto_original", "INTEGER", mode="REQUIRED", description="Chave natural: EAN do produto"),
    SchemaField("cod_prod_catarinense","INTEGER", mode="NULLABLE", description="Codigo interno do produto Catarinense"),
    SchemaField("valor_produto",        "INTEGER", mode="REQUIRED", description="Volume SO Preco Popular (proxy de valor)"),
    SchemaField("competencia",          "STRING",  mode="NULLABLE", description="Competencia de referencia (MM/AAAA)"),
    SchemaField("data_inicio_validade", "DATE",    mode="REQUIRED", description="Data de inicio da validade do registro"),
    SchemaField("data_fim_validade",    "DATE",    mode="NULLABLE", description="Data de fim da validade (NULL = ativo)"),
    SchemaField("flag_ativo",           "BOOLEAN", mode="REQUIRED", description="TRUE = registro atual, FALSE = historico"),
    SchemaField("dt_carga",             "TIMESTAMP",mode="REQUIRED",description="Timestamp da carga ETL"),
]


def ensure_table_exists(client: bigquery.Client, table_ref: str, schema: list) -> bigquery.Table:
    """Cria a tabela no BigQuery caso não exista. Idempotente."""
    table = bigquery.Table(table_ref, schema=schema)
    table = client.create_table(table, exists_ok=True)
    print(f"Tabela pronta: {table.full_table_id}")
    return table


gold_table = ensure_table_exists(bq_client, BQ_TABLE_REF, GOLD_SCHEMA)

Tabela pronta: proc-de-dados-iqvia-com-scd:conj_dados_iqvia.dim_produto_scd2


## 5. Extração — Leitura do Parquet Silver

In [45]:
def read_parquet_from_gcs(bucket: storage.Bucket, blob_path: str) -> pd.DataFrame:
    """Lê um arquivo Parquet diretamente do GCS e retorna um DataFrame."""
    blob = bucket.blob(blob_path)
    data = blob.download_as_bytes()
    return pd.read_parquet(io.BytesIO(data))


bucket      = gcs_client.bucket(GCS_BUCKET_NAME)
silver_path = f"{GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/{SILVER_FILE_NAME}"

df_silver = read_parquet_from_gcs(bucket, silver_path)

print(f"Silver lido: {silver_path}")
print(f"Shape: {df_silver.shape}")
display(df_silver.head())

Silver lido: silver/iqvia/2026/03/20/vendas_enriquecido_v2.parquet
Shape: (40, 11)


,cod_brick,brick,ean,cod_prod_catarinense,vol_si_concorrente,vol_so_concorrente,vol_so_preco_popular,competencia,dt_processamento,cod_filial,desc_brick
0,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,950,CAMPO GRANDE - CENTRO
1,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,373,CAMPO GRANDE - CENTRO
2,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,386,CAMPO GRANDE - CENTRO
3,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,478,CAMPO GRANDE - CENTRO
4,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,672,CAMPO GRANDE - CENTRO


## 6. Preparação dos Dados de Entrada (Staging)

In [46]:
def generate_sk(ean: int, valor: int, data_inicio: date) -> str:
    """Gera a Surrogate Key (SHA256) com base nos atributos que identificam univocamente uma versão do produto."""
    raw = f"{ean}|{valor}|{data_inicio.isoformat()}"
    return hashlib.sha256(raw.encode()).hexdigest()


def prepare_staging(df: pd.DataFrame, load_date: date, load_timestamp: datetime) -> pd.DataFrame:
    """Seleciona e renomeia as colunas para o formato da tabela Gold.

    Como um mesmo EAN pode aparecer em múltiplas filiais, agrupamos pelo produto
    somando os volumes para obter o valor consolidado por competência.
    """
    df_staged = (
        df.groupby(["ean", "cod_prod_catarinense", "competencia"], as_index=False)
        .agg(valor_produto=("vol_so_preco_popular", "sum"))
    )

    df_staged = df_staged.rename(columns={"ean": "id_produto_original"})
    df_staged["id_produto_original"]  = df_staged["id_produto_original"].astype(int)
    df_staged["cod_prod_catarinense"] = df_staged["cod_prod_catarinense"].astype(int)
    df_staged["valor_produto"]        = df_staged["valor_produto"].astype(int)
    df_staged["data_inicio_validade"] = load_date
    df_staged["data_fim_validade"]    = None
    df_staged["flag_ativo"]           = True
    df_staged["dt_carga"]             = load_timestamp

    df_staged["sk_produto"] = df_staged.apply(
        lambda r: generate_sk(r["id_produto_original"], r["valor_produto"], load_date),
        axis=1
    )

    return df_staged[[
        "sk_produto", "id_produto_original", "cod_prod_catarinense",
        "valor_produto", "competencia",
        "data_inicio_validade", "data_fim_validade", "flag_ativo", "dt_carga"
    ]]


df_staging = prepare_staging(df_silver, LOAD_DATE, LOAD_TIMESTAMP)

print(f"Staging shape: {df_staging.shape}")
print(f"Produtos únicos: {df_staging['id_produto_original'].nunique()}")
display(df_staging)

Staging shape: (8, 9)
Produtos únicos: 8


,sk_produto,id_produto_original,cod_prod_catarinense,valor_produto,competencia,data_inicio_validade,data_fim_validade,flag_ativo,dt_carga
0,55c6e531aea6d736da708bc4627b4a56654dc4a8becedc...,32689150,741013,25,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00
1,c83a564607f891a1ba85dfbb5e838c3d7ff72aad4d1de9...,42110200,630693,25,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00
2,3c60479302edea35126c0dc063d8402115779109e221f9...,42277217,739189,115,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00
3,914bfcd86b8ed118aa9fdcdb881c95c212df2139d4fb79...,42355014,735367,5,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00
4,a93c97fdd2f51cb52990b1882c58659e25e40c63cbeb82...,42355465,735344,0,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00
5,e25ec89fa0710c08cdc1e4ba63bd44c4fc2b26ed9557fd...,42360407,734398,75,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00
6,dba6dbf0a6826782e75b1f1f73735a8b6667652819f509...,42360414,734399,4995,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00
7,2689e1e935d5aa7f8b19cb266cf60cfa048d9643ce7b87...,42389248,736535,35,12/2022,2026-03-20,None,True,2026-03-20 13:00:02.291308+00:00


## 7. SCD Tipo 2 — Lógica de Merge

```
Para cada produto no staging:
  ├── Existe na tabela Gold (flag_ativo = TRUE)?
  │     ├── NÃO  → Inserir como novo registro ativo
  │     └── SIM  → O valor_produto mudou?
  │               ├── NÃO  → Ignorar (sem alteração)
  │               └── SIM  → Fechar registro antigo (flag_ativo=FALSE, data_fim=hoje)
  │                          Inserir novo registro ativo
```

In [47]:
def read_active_records(client: bigquery.Client, table_ref: str) -> pd.DataFrame:
    """Lê os registros ativos (flag_ativo = TRUE) da tabela Gold no BigQuery."""
    query = f"""
        SELECT
            sk_produto,
            id_produto_original,
            valor_produto,
            data_inicio_validade
        FROM `{table_ref}`
        WHERE flag_ativo = TRUE
    """
    df = client.query(query).to_dataframe()
    print(f"Registros ativos na tabela Gold: {len(df)}")
    return df


df_current = read_active_records(bq_client, BQ_TABLE_REF)
display(df_current)

Registros ativos na tabela Gold: 9


d:\analise-de-dados-devinhouse\modulo_02_arquitetura_e_visualizacao_de_dados\mini_projeto\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,sk_produto,id_produto_original,valor_produto,data_inicio_validade
0,6c80b537646263fa73edfa6ae377ee4519309c6f1897c4...,42176763,0,2026-03-20
1,a93c97fdd2f51cb52990b1882c58659e25e40c63cbeb82...,42355465,0,2026-03-20
2,914bfcd86b8ed118aa9fdcdb881c95c212df2139d4fb79...,42355014,5,2026-03-20
3,55c6e531aea6d736da708bc4627b4a56654dc4a8becedc...,32689150,25,2026-03-20
4,c83a564607f891a1ba85dfbb5e838c3d7ff72aad4d1de9...,42110200,25,2026-03-20
5,2689e1e935d5aa7f8b19cb266cf60cfa048d9643ce7b87...,42389248,35,2026-03-20
6,e25ec89fa0710c08cdc1e4ba63bd44c4fc2b26ed9557fd...,42360407,75,2026-03-20
7,3c60479302edea35126c0dc063d8402115779109e221f9...,42277217,115,2026-03-20
8,e010dfc3f9f2eebd77d9d6e56a35da8863971161469600...,42360414,210,2026-03-20


In [48]:
def apply_scd2(
    df_staging: pd.DataFrame,
    df_current: pd.DataFrame,
    load_date: date,
    load_timestamp: datetime
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Aplica a lógica SCD Tipo 2 e retorna:
        - df_to_insert : novos registros a inserir (novos + versões atualizadas)
        - df_to_close  : SKs de registros ativos a serem encerrados

    Casos tratados:
        1. Produto novo (não existe na Gold)         → Inserir como ativo
        2. Produto com valor alterado                → Fechar antigo + inserir novo
        3. Produto sem alteração                     → Ignorar
        4. Produto ativo na Gold ausente no staging  → Fechar (exclusão lógica)
    """
    to_insert   = []
    sk_to_close = []

    staging_ids = set(df_staging["id_produto_original"].tolist())

    current_map = {
        int(row["id_produto_original"]): row
        for _, row in df_current.iterrows()
    }

    # Casos 1, 2 e 3: percorre o staging
    for _, new_row in df_staging.iterrows():
        prod_id = int(new_row["id_produto_original"])

        if prod_id not in current_map:
            # Caso 1 — produto novo
            to_insert.append(new_row.to_dict())

        else:
            existing = current_map[prod_id]
            if int(existing["valor_produto"]) != int(new_row["valor_produto"]):
                # Caso 2 — valor alterado
                sk_to_close.append(existing["sk_produto"])
                to_insert.append(new_row.to_dict())
            # Caso 3 — sem alteração: ignora

    # Caso 4 — produto ativo na Gold que sumiu do staging (exclusão lógica)
    for prod_id, existing in current_map.items():
        if prod_id not in staging_ids:
            sk_to_close.append(existing["sk_produto"])

    df_to_insert = pd.DataFrame(to_insert) if to_insert else pd.DataFrame()
    df_to_close  = pd.DataFrame({"sk_produto": sk_to_close}) if sk_to_close else pd.DataFrame()

    novos      = sum(1 for _, r in df_staging.iterrows() if int(r["id_produto_original"]) not in current_map)
    alterados  = len(sk_to_close) - (len(df_current) - len([p for p in current_map if p in staging_ids and int(current_map[p]["valor_produto"]) == next((int(r["valor_produto"]) for _, r in df_staging.iterrows() if int(r["id_produto_original"]) == p), -1)]))
    removidos  = sum(1 for p in current_map if p not in staging_ids)

    print(f"Produtos no staging          : {len(df_staging)}")
    print(f"Registros ativos na Gold     : {len(df_current)}")
    print(f"  ↳ Novos a inserir          : {novos}")
    print(f"  ↳ Valores alterados        : {len(sk_to_close) - removidos}")
    print(f"  ↳ Exclusoes logicas        : {removidos}")
    print(f"  ↳ Sem alteracao (ignorados): {len(df_staging) - novos - (len(sk_to_close) - removidos)}")
    print(f"Total a inserir              : {len(df_to_insert)}")
    print(f"Total a fechar               : {len(df_to_close)}")

    return df_to_insert, df_to_close


df_to_insert, df_to_close = apply_scd2(df_staging, df_current, LOAD_DATE, LOAD_TIMESTAMP)

Produtos no staging          : 8
Registros ativos na Gold     : 9
  ↳ Novos a inserir          : 0
  ↳ Valores alterados        : 1
  ↳ Exclusoes logicas        : 1
  ↳ Sem alteracao (ignorados): 7
Total a inserir              : 1
Total a fechar               : 2


## 8. Carga — Aplicando as Mudanças no BigQuery

In [49]:
def close_records(client: bigquery.Client, table_ref: str, sk_list: list[str], close_date: date) -> int:
    """Encerra registros antigos: seta flag_ativo=FALSE e data_fim_validade."""
    if not sk_list:
        print("Nenhum registro para fechar.")
        return 0

    sk_values = ", ".join(f"'{sk}'" for sk in sk_list)
    query = f"""
        UPDATE `{table_ref}`
        SET
            flag_ativo        = FALSE,
            data_fim_validade = DATE '{close_date.isoformat()}'
        WHERE sk_produto IN ({sk_values})
          AND flag_ativo = TRUE
    """
    job = client.query(query)
    job.result()  # aguarda conclusão
    print(f"Registros encerrados: {job.num_dml_affected_rows}")
    return job.num_dml_affected_rows


rows_closed = close_records(
    bq_client,
    BQ_TABLE_REF,
    df_to_close["sk_produto"].tolist() if not df_to_close.empty else [],
    LOAD_DATE
)

Registros encerrados: 2


In [50]:
def insert_records(client: bigquery.Client, table_ref: str, df: pd.DataFrame) -> int:
    """Insere novos registros na tabela Gold do BigQuery."""
    if df.empty:
        print("Nenhum registro novo para inserir.")
        return 0

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        schema_update_options=[],
    )

    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()  # aguarda conclusão
    print(f"Registros inseridos: {len(df)}")
    return len(df)


rows_inserted = insert_records(bq_client, BQ_TABLE_REF, df_to_insert)

d:\analise-de-dados-devinhouse\modulo_02_arquitetura_e_visualizacao_de_dados\mini_projeto\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Registros inseridos: 1


## 9. Relatório Final da Carga

In [51]:
print("=" * 60)
print("RELATORIO DE CARGA — CAMADA GOLD (SCD TIPO 2)")
print("=" * 60)
print(f"  Pipeline         : IQVIA ETL")
print(f"  Tabela BigQuery  : {BQ_TABLE_REF}")
print(f"  Silver lida      : {GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/")
print(f"  Timestamp        : {LOAD_TIMESTAMP.strftime('%d/%m/%Y %H:%M:%S UTC')}")
print("-" * 60)
print(f"  Produtos no staging  : {len(df_staging)}")
print(f"  Registros inseridos  : {rows_inserted}")
print(f"  Registros encerrados : {rows_closed}")
print(f"  Sem alteracao        : {len(df_staging) - rows_inserted}")
print("=" * 60)

RELATORIO DE CARGA — CAMADA GOLD (SCD TIPO 2)
  Pipeline         : IQVIA ETL
  Tabela BigQuery  : proc-de-dados-iqvia-com-scd.conj_dados_iqvia.dim_produto_scd2
  Silver lida      : silver/iqvia/2026/03/20/
  Timestamp        : 20/03/2026 13:00:02 UTC
------------------------------------------------------------
  Produtos no staging  : 8
  Registros inseridos  : 1
  Registros encerrados : 2
  Sem alteracao        : 7


## 10. Verificação — Estado Final da Tabela Gold

In [52]:
query_verify = f"""
    SELECT
        sk_produto,
        id_produto_original,
        valor_produto,
        competencia,
        FORMAT_DATE('%d/%m/%Y', data_inicio_validade) AS data_inicio_validade,
        FORMAT_DATE('%d/%m/%Y', data_fim_validade)    AS data_fim_validade,
        flag_ativo
    FROM `{BQ_TABLE_REF}`
    ORDER BY id_produto_original, data_inicio_validade DESC
"""

df_final = bq_client.query(query_verify).to_dataframe()

ativos    = df_final[df_final["flag_ativo"] == True].shape[0]
historico = df_final[df_final["flag_ativo"] == False].shape[0]

print(f"Total de registros na tabela : {len(df_final)}")
print(f"  Registros ativos           : {ativos}")
print(f"  Registros historicos       : {historico}")
print()
display(df_final)

print("\nCarga Gold finalizada. Pipeline ETL IQVIA completo.")

Total de registros na tabela : 10
  Registros ativos           : 8
  Registros historicos       : 2



d:\analise-de-dados-devinhouse\modulo_02_arquitetura_e_visualizacao_de_dados\mini_projeto\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,sk_produto,id_produto_original,valor_produto,competencia,data_inicio_validade,data_fim_validade,flag_ativo
0,55c6e531aea6d736da708bc4627b4a56654dc4a8becedc...,32689150,25,12/2022,20/03/2026,None,True
1,c83a564607f891a1ba85dfbb5e838c3d7ff72aad4d1de9...,42110200,25,12/2022,20/03/2026,None,True
2,6c80b537646263fa73edfa6ae377ee4519309c6f1897c4...,42176763,0,12/2022,20/03/2026,20/03/2026,False
3,3c60479302edea35126c0dc063d8402115779109e221f9...,42277217,115,12/2022,20/03/2026,None,True
4,914bfcd86b8ed118aa9fdcdb881c95c212df2139d4fb79...,42355014,5,12/2022,20/03/2026,None,True
5,a93c97fdd2f51cb52990b1882c58659e25e40c63cbeb82...,42355465,0,12/2022,20/03/2026,None,True
6,e25ec89fa0710c08cdc1e4ba63bd44c4fc2b26ed9557fd...,42360407,75,12/2022,20/03/2026,None,True
7,dba6dbf0a6826782e75b1f1f73735a8b6667652819f509...,42360414,4995,12/2022,20/03/2026,None,True
8,e010dfc3f9f2eebd77d9d6e56a35da8863971161469600...,42360414,210,12/2022,20/03/2026,20/03/2026,False
9,2689e1e935d5aa7f8b19cb266cf60cfa048d9643ce7b87...,42389248,35,12/2022,20/03/2026,None,True



Carga Gold finalizada. Pipeline ETL IQVIA completo.
